In [1]:
# 查看当前挂载的数据集目录, 该目录下的变更重启环境后会自动还原
# View dataset directory. 
# This directory will be recovered automatically after resetting environment. 
!ls /home/aistudio/data

tables.sql  test.json  test2.json  train.json


In [2]:
# 查看工作区文件，该目录下除data目录外的变更将会持久保存。请及时清理不必要的文件，避免加载过慢。
# View personal work directory. 
# All changes, except /data, under this directory will be kept even after reset. 
# Please clean unnecessary files in time to speed up environment loading. 
!ls /home/aistudio

data		    main.ipynb				submission.csv
external-libraries  new-workspace.jupyterlab-workspace	work


In [3]:
# 如果需要进行持久化安装, 需要使用持久化路径, 如下方代码示例:
# If a persistence installation is required, 
# you need to use the persistence path as the following: 
!mkdir /home/aistudio/external-libraries
!pip install beautifulsoup4 -t /home/aistudio/external-libraries

mkdir: cannot create directory '/home/aistudio/external-libraries': File exists
ERROR: Can not combine '--user' and '--target'


In [4]:
# 同时添加如下代码, 这样每次环境(kernel)启动的时候只要运行下方代码即可: 
# Also add the following code, 
# so that every time the environment (kernel) starts, 
# just run the following code: 
import sys 
sys.path.append('/home/aistudio/external-libraries')

In [5]:
import os
import json
import paddle
import requests
from paddlenlp.transformers import AutoTokenizer, AutoModelForCausalLM

/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [6]:
# 加载表结构数据
def load_table_schema(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content

In [7]:
# 构建prompt模板
# 原版
# def build_prompt(nl_question, table_schemas):
#     if isinstance(table_schemas, list):
#         table_schemas = "\n".join(table_schemas)
#     prompt = f"""根据以下自然语言问题和数据库表结构生成对应的SQL查询。
#         只返回SQL语句，不要复述问题，不要添加解释，不要包含其他内容。
#         以以下格式返回（不要添加多余字符）：
#         查询SQL: SELECT ... FROM ... WHERE ...;

#         问题：
#         {nl_question}

#         数据库表结构：
#         {table_schemas if table_schemas else '无表结构信息'}
#         """
#     return prompt


# yb
# def build_prompt(nl_question, table_schemas):
#     optimized_prompt = f"""请根据以下问题和表结构生成最优化的SQL查询，遵循以下规则：
# 1. 使用大写关键字（SELECT/WHERE/JOIN）
# 2. 显式指定列名（避免SELECT *）
# 3. 优先使用索引列（如gmsfhm）
# 4. 准确匹配表别名（如graph_tag_idcard_base AS a）
# 5. 添加必要的JOIN条件
# 6. 对文本条件使用单引号
# 7. 包含分号结尾
# 8. 使用EXISTS代替IN子查询
# 9. 避免在WHERE条件列使用函数

# 表结构说明：
# {format_schemas(table_schemas)}

# 问题特征分析：
# "{nl_question}" 需要关联以下要素：
# - 查询目标字段：{"[自动提取关键字段]"}
# - 关联表：{"[自动匹配相关表]"}
# - 过滤条件：{"[解析条件表达式]"}
# - 聚合需求：{"[识别COUNT/SUM等]"}
# - 排序要求：{"[识别ORDER BY]"}
# - 性能要点：{"[识别高频查询模式]"} 

# 请生成满足以下要求的SQL：
# 1. 语法100%正确
# 2. 使用最优执行路径
# 3. 包含必要索引提示
# 4. 正确关联所有表
# 5. 结果无歧义

# 最终SQL："""
#     return optimized_prompt

# g
def build_prompt(nl_question, table_schemas):
    """
    生成精简的 Text2SQL 提示，仅输出 SQL 语句。
    """
    # 将表结构列表合并为字符串
    if isinstance(table_schemas, list):
        table_schemas = "\n".join(table_schemas)

    prompt = (
        "### 任务：将以下自然语言问题转换为 SQL，只输出 SQL 语句，不要任何解释和注释。\n"
        "### 格式示例：SELECT column1, column2 FROM table WHERE condition;\n\n"
        "问题：\n"
        f"```\n{nl_question}\n```\n\n"
        "表结构：\n"
        f"```\n{table_schemas or '无'}\n```"
    )
    return prompt

In [8]:
# 从SQL文件中提取表结构信息
def extract_table_schemas(sql_content, num_tables=3):
    # 分割SQL文件内容，获取前几个表的定义
    tables = []
    table_definitions = sql_content.split(';')
    
    count = 0
    schema_text = ""
    
    for table_def in table_definitions:
        if 'CREATE TABLE' in table_def and count < num_tables:
            # 清理并格式化表定义
            cleaned_def = table_def.strip()
            if cleaned_def:
                schema_text += cleaned_def + ";\n\n"
                count += 1
    
    return schema_text

In [9]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="float16")

(…)wen2.5-7B-Instruct/tokenizer_config.json: 100%|██████████| 7.30k/7.30k [00:00<00:00, 17.0MB/s]
(…)nity/Qwen/Qwen2.5-7B-Instruct/vocab.json: 100%|██████████| 2.78M/2.78M [00:00<00:00, 283MB/s]
(…)nity/Qwen/Qwen2.5-7B-Instruct/merges.txt: 100%|██████████| 1.67M/1.67M [00:00<00:00, 362MB/s]
[2025-05-08 20:34:45,547] [    INFO] - The `unk_token` parameter needs to be defined: we use `eos_token` by default.
(…)ity/Qwen/Qwen2.5-7B-Instruct/config.json: 100%|██████████| 657/657 [00:00<00:00, 2.45MB/s]
[2025-05-08 20:34:45,938] [    INFO] - We are using <class 'paddlenlp.transformers.qwen2.modeling.Qwen2ForCausalLM'> to load 'Qwen/Qwen2.5-7B-Instruct'.
[2025-05-08 20:34:45,938] [    INFO] - Loading configuration file /home/aistudio/.paddlenlp/models/Qwen/Qwen2.5-7B-Instruct/config.json
(…)7B-Instruct/model.safetensors.index.json: 100%|██████████| 27.8k/27.8k [00:00<00:00, 60.2MB/s]
[2025-05-08 20:34:45,961] [    INFO] - Loading weights file from cache at /home/aistudio/.paddlenlp/models/Qwe

In [15]:
def generate_sql_with_qwen(prompt):
    try:
        inputs = tokenizer(prompt, return_tensors="pd")
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.7,
            repetition_penalty=1.1
        )
        output_ids = outputs[0][0].tolist()
        sql = tokenizer.decode(output_ids, skip_special_tokens=True)
        if prompt in sql:
            sql = sql[len(prompt):].strip()
        try:
            sql = sql.rsplit("查询SQL:", 1)[1].strip()
            return sql
        except:
            return sql
    except Exception as e:
        print(f"使用千问模型生成SQL时出错: {e}")
        import traceback
        traceback.print_exc()
        return None

In [16]:
# 加载测试数据
tables_path = "data/tables.sql"
tables_data = load_table_schema(tables_path)
    
# 提取表结构信息
table_schemas = extract_table_schemas(tables_data)

In [17]:
# 读取测试数据集
test_data_path = 'data/test.json'  # 替换为你的实际路径
with open(test_data_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
from tqdm import tqdm
results = []
for item in tqdm(data, desc="Processing", unit="sample"):
    example_id = item.get('id')
    nl_input = item.get('NL')
    prompt = build_prompt(nl_input, table_schemas)
    pred_sql = generate_sql_with_qwen(prompt)
    results.append({
        'id': example_id,
        'pred_sql': pred_sql
    })

Processing:  59%|█████▉    | 592/1000 [46:30<27:02,  3.98s/sample]  

In [14]:
# 写入 CSV 文件
import csv
csv_file_path = 'submission.csv'
with open(csv_file_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'pred_sql'])
    writer.writeheader()
    writer.writerows(results)
print(f"成功写入 {csv_file_path}，共 {len(results)} 条记录。")

成功写入 submission.csv，共 1000 条记录。


请点击[此处](https://ai.baidu.com/docs#/AIStudio_Project_Notebook/a38e5576)查看本环境基本用法.  <br>
Please click [here ](https://ai.baidu.com/docs#/AIStudio_Project_Notebook/a38e5576) for more detailed instructions. 